In [ ]:
# BUY RULE:
# if (bm > 1.5 AND mom12m > 1.0) OR mve > 1.5:
#     signal = BUY

# SELL RULE:
# if bm < -1.5 OR mom12m < -1.0:
#     signal = SELL

In [3]:
import pandas as pd
import numpy as np

# 1. Load quantitative data
green_cleaned = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
green_cleaned['ret_fwd_1'] = (green_cleaned.groupby('permno')['ret_excess'].shift(-1))

In [4]:
# 2. Create year_month column to align with FinBERT data format
green_cleaned['datadate'] = pd.to_datetime(green_cleaned['datadate'])
green_cleaned['year_month'] = green_cleaned['datadate'].dt.strftime('%Y-%m')

# 3. Load and format FinBERT sentiment data
df_sent = pd.read_csv('../examples/monthly_signals_decay.csv')
df_sent = df_sent.rename(columns={
    'symbol': 'ticker',
    'weighted_sentiment_score': 'finbert_score'
})
df_sent_subset = df_sent[['ticker', 'year_month', 'finbert_score']]

# 4. Merge datasets and handle missing news
green_cleaned = pd.merge(green_cleaned, df_sent_subset, on=['ticker', 'year_month'], how='left')
green_cleaned['finbert_score'] = green_cleaned['finbert_score'].fillna(0.0)

# Convert datadate back to string so your start_date filter below works perfectly
green_cleaned['datadate'] = green_cleaned['datadate'].dt.strftime('%Y-%m-%d')

# 5. Add finbert_score to your variables list
variables = ['datadate', 'permno', 'ticker', 'comnam', 'bm', 'mom12m', 'mve', 'finbert_score', 'ret_fwd_1']

In [9]:
start_date = '2024-01-31'

analysis_df = green_cleaned[
    (green_cleaned['datadate'] == start_date)
][variables].copy()

# BUY RULE:
# if bm > 0.65 AND (finbert_score > 0.15 AND finbert_score < 0.3):
#     signal = BUY

# SELL RULE:
# if bm < -0.6 OR finbert_score < -0.1:
#     signal = SELL

conditions = [
    ( (analysis_df['bm'] > 0.65) & ( (analysis_df['finbert_score'] > 0.15) & (analysis_df['finbert_score'] < 0.3))),
    ( (analysis_df['bm'] < -0.6) | (analysis_df['finbert_score'] < -0.1) ),
]

# 3. Define choices
choices = ['BUY', 'SELL']

# 4. Create the 'signal' column
analysis_df['signal'] = np.select(conditions, choices, default='HOLD')

buys = analysis_df[analysis_df['signal'] == 'BUY'][['datadate', 'permno', 'ticker', 'comnam', 'ret_fwd_1']]
sells = analysis_df[analysis_df['signal'] == 'SELL'][['datadate', 'permno', 'ticker', 'comnam', 'ret_fwd_1']]


print("\n--- BUY ---")
with pd.option_context('display.max_rows', None):
    print(buys)


print("\n--- SELL ---")
print(sells)

buys.to_csv('buys_02.csv', index=False)
sells.to_csv('sells_02.csv', index=False)


--- BUY ---
          datadate  permno ticker                   comnam  ret_fwd_1
248992  2024-01-31   86868     GS  GOLDMAN SACHS GROUP INC   0.016086

--- SELL ---
          datadate  permno ticker                       comnam  ret_fwd_1
826     2024-01-31   10104   ORCL                  ORACLE CORP  -0.004379
1193    2024-01-31   10107   MSFT               MICROSOFT CORP   0.038081
2195    2024-01-31   10145    HON  HONEYWELL INTERNATIONAL INC  -0.016313
8796    2024-01-31   11308     KO                 COCA COLA CO   0.004709
9166    2024-01-31   11403   CDNS   CADENCE DESIGN SYSTEMS INC   0.050990
...            ...     ...    ...                          ...        ...
267630  2024-01-31   92611      V                     VISA INC   0.032029
268730  2024-01-31   93089   VRSK         VERISK ANALYTICS INC  -0.002668
268875  2024-01-31   93096     DG      DOLLAR GENERAL CORP NEW   0.096050
268950  2024-01-31   93132   FTNT                 FORTINET INC   0.067439
269243  2024-01-31 

In [10]:
start_date = '2024-02-29'

# BUY RULE:
# if (bm > 1.2 AND finbert_score > 0.05) OR (mom12m > 0.85 AND finbert_score > 0.0):
#     signal = BUY

# SELL RULE:
# if (bm < -1.25 OR finbert_score < -0.15):
#     signal = SELL

analysis_df = green_cleaned[
    (green_cleaned['datadate'] == start_date)
][variables].copy()

conditions = [
    ( (analysis_df['bm'] > 1.2) & (analysis_df['finbert_score'] > 0.05) ) | ( (analysis_df['mom12m'] > 0.85) & (analysis_df['finbert_score'] > 0.0) ),

    ( (analysis_df['bm'] < -1.25) | (analysis_df['finbert_score'] < -0.15) )
]

# 3. Define choices
choices = ['BUY', 'SELL']

# 4. Create the 'signal' column
analysis_df['signal'] = np.select(conditions, choices, default='HOLD')

buys = analysis_df[analysis_df['signal'] == 'BUY'][['datadate', 'permno', 'ticker', 'comnam', 'ret_fwd_1']]
sells = analysis_df[analysis_df['signal'] == 'SELL'][['datadate', 'permno', 'ticker', 'comnam', 'ret_fwd_1']]


print("\n--- BUY ---")
with pd.option_context('display.max_rows', None):
    print(buys)


print("\n--- SELL ---")
print(sells)

buys.to_csv('buys_03.csv', index=False)
sells.to_csv('sells_03.csv', index=False)


--- BUY ---
          datadate  permno ticker                        comnam  ret_fwd_1
1194    2024-02-29   10107   MSFT                MICROSOFT CORP   0.012816
23693   2024-02-29   13407   META            META PLATFORMS INC  -0.013583
31904   2024-02-29   14714   ANET           ARISTA NETWORKS INC   0.040522
33008   2024-02-29   14889    CFG  CITIZENS FINANCIAL GROUP INC   0.151801
56488   2024-02-29   19502    WBA  WALGREENS BOOTS ALLIANCE INC   0.015926
158111  2024-02-29   50876    LLY                LILLY ELI & CO   0.027915
178226  2024-02-29   59328   INTC                    INTEL CORP   0.021716
225573  2024-02-29   79094    JBL                     JABIL INC  -0.074673
227709  2024-02-29   79698     IT                   GARTNER INC   0.019564
234996  2024-02-29   82598   NTAP                    NETAPP INC   0.173550
248993  2024-02-29   86868     GS       GOLDMAN SACHS GROUP INC   0.069315
261761  2024-02-29   90455   MPWR        MONOLITHIC PWR SYS INC  -0.061755

--- SELL --

In [11]:
start_date = '2024-03-31'

# BUY RULE:
# if (bm > 1.2 AND mom12m < 0.5) OR (bm > 1.2 AND mom12m > 0.5 AND finbert_score > 0.0 AND mom12m > 0.7) OR (finbert_score = 0.0 AND bm > 1.2 AND mom12m < 0.5 AND mom12m > 0.2):
#     signal = BUY

# SELL RULE:
# if bm < -1.2 OR finbert_score < -0.03 OR (bm > 1.0 AND finbert_score < -0.06) OR (mom12m < -0.5 AND mve > 1.0 AND finbert_score < -0.04):
#     signal = SELL

analysis_df = green_cleaned[
    (green_cleaned['datadate'] == start_date)
][variables].copy()

conditions = [
    ( (analysis_df['bm'] > 1.2) & (analysis_df['mom12m'] < 0.5) ) | ( (analysis_df['bm'] > 1.2) & (analysis_df['mom12m'] > 0.5) & (analysis_df['finbert_score'] > 0.0) & (analysis_df['mom12m'] > 0.7) ) | ( (analysis_df['finbert_score'] == 0.0) & (analysis_df['bm'] > 1.2) & (analysis_df['mom12m'] < 0.5) & (analysis_df['mom12m'] > 0.2) ),
    ( (analysis_df['bm'] < -1.2) | (analysis_df['finbert_score'] < -0.03) ) | ( (analysis_df['bm'] > 1.0) & (analysis_df['finbert_score'] < -0.06) ) | ( (analysis_df['mom12m'] < -0.5) & (analysis_df['mve'] > 1.0) & (analysis_df['finbert_score'] < -0.04) ),
]

# 3. Define choices
choices = ['BUY', 'SELL']

# 4. Create the 'signal' column
analysis_df['signal'] = np.select(conditions, choices, default='HOLD')

buys = analysis_df[analysis_df['signal'] == 'BUY'][['datadate', 'permno', 'ticker', 'comnam', 'ret_fwd_1']]
sells = analysis_df[analysis_df['signal'] == 'SELL'][['datadate', 'permno', 'ticker', 'comnam', 'ret_fwd_1']]


print("\n--- BUY ---")
with pd.option_context('display.max_rows', None):
    print(buys)


print("\n--- SELL ---")
print(sells)

buys.to_csv('buys_04.csv', index=False)
sells.to_csv('sells_04.csv', index=False)


--- BUY ---
          datadate  permno ticker                            comnam  ret_fwd_1
17571   2024-03-31   12369     GM                 GENERAL MOTORS CO  -0.022782
20135   2024-03-31   12558    KMI                 KINDER MORGAN INC   0.007705
24681   2024-03-31   13688    PCG                      P G & E CORP   0.016183
32348   2024-03-31   14776    SYF               SYNCHRONY FINANCIAL   0.015244
33009   2024-03-31   14889    CFG      CITIZENS FINANCIAL GROUP INC  -0.053198
35021   2024-03-31   15408    KHC                    KRAFT HEINZ CO   0.041641
37439   2024-03-31   15707    HPE     HEWLETT PACKARD ENTERPRISE CO  -0.045873
41535   2024-03-31   16851     DD             DUPONT DE NEMOURS INC  -0.059089
56489   2024-03-31   19502    WBA      WALGREENS BOOTS ALLIANCE INC  -0.187273
58938   2024-03-31   20057   VTRS                       VIATRIS INC  -0.035688
67168   2024-03-31   21573     IP            INTERNATIONAL PAPER CO  -0.109262
76008   2024-03-31   22976    WBD      